# Preparing a Dataset for Analysis

Before any analysis can begin, the underlying data must be **understood, cleaned, and shaped** into a form that supports reliable conclusions. Skipping this step is one of the most common causes of incorrect results, misleading dashboards, and wasted effort.

> **Note:** This module uses **SQL** throughout. SQL is the most cross-compute compatible language — it runs on classic clusters, serverless SQL warehouses, and virtually every database engine, making the techniques here portable regardless of your platform.

> **Data disclaimer:** All data used in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals. No real data has been used.

## Why Does Messy Data Matter?

Most data quality problems don't cause your query to fail — they cause it to return a result that is **plausible but incorrect**. A headcount inflated by 25%, a region split into three groups instead of one, or a pupil quietly dropped from a join. No error messages, just wrong answers that look right.

### Problem 1: Duplicates inflate your counts

If the same record appears more than once, every `COUNT`, `SUM`, and `AVG` is affected. The `pupils_autumn_2024` table in our messy dataset contains both exact duplicates and natural key conflicts. The query below groups by school and compares the raw row count to the distinct pupil count — any difference is inflation caused by duplicates.

In [0]:
USE CATALOG catalog_40_copper_analyst_training;

In [0]:
-- The pupils_autumn_2024 table contains duplicate rows
-- Compare raw count to distinct pupil count per school to reveal inflation
SELECT
  school_urn
  ,COUNT(*) AS reported_headcount
  ,COUNT(DISTINCT pupil_id) AS actual_headcount
  ,COUNT(*) - COUNT(DISTINCT pupil_id) AS inflation
FROM bronze.pupils_autumn_2024
GROUP BY school_urn
ORDER BY inflation DESC;

### Problem 2: Inconsistent labels fragment your groups

If the same value is recorded in different ways — different casing, abbreviations, or placeholder values — a `GROUP BY` treats each variant as a separate category. The `schools_autumn_2024` table records school type inconsistently: `'Academy'`, `'academy'`, `'Acad'`, and `'N/A'` all appear. The query below shows how what should be two or three groups becomes six.

In [0]:
-- The schools_autumn_2024 table has inconsistent school_type values
-- GROUP BY treats each variant as a separate category
SELECT
  school_type
  ,COUNT(*) AS school_count
FROM bronze.schools_autumn_2024
GROUP BY school_type
ORDER BY school_type;

### Problem 3: Fragmented data loses records silently

When reference data is incomplete, joins can **silently drop rows**. The `pupils_autumn_2024` table contains a pupil (P019) enrolled at school URN 999999 — a school that doesn’t exist in `schools_autumn_2024`. An inner join quietly removes that pupil from the results with no error or warning.

In [0]:
-- How many distinct pupils exist vs how many survive an inner join to schools?
-- The difference is pupils silently lost due to missing school references
SELECT
  'Total distinct pupils' AS metric,
  COUNT(DISTINCT pupil_id) AS count
FROM bronze.pupils_autumn_2024

UNION ALL

SELECT
  'Pupils after INNER JOIN to schools',
  COUNT(DISTINCT p.pupil_id)
FROM bronze.pupils_autumn_2024 p
INNER JOIN bronze.schools_autumn_2024 s
  ON p.school_urn = s.school_urn;

## The Benefits of Data Preparation

Investing time upfront makes every downstream step faster, cheaper, and more reliable:

| Benefit | Why it matters |
| --- | --- |
| **Accuracy** | Duplicates, fragmented labels, and orphaned keys silently distort counts, sums, and averages. Clean data means trustworthy numbers. |
| **Consistency** | A single preparation step guarantees every query, report, and dashboard applies the same transformations — eliminating analyst-to-analyst variation. |
| **Documentation** | The cleaning process creates an auditable trail of what was included, excluded, and why. |
| **Time saving** | Debugging unexpected results, re-running queries, and fielding stakeholder questions costs far more than preparing data once. |
| **Cost saving** | Duplicates inflate storage and scan costs. Unstandardised fields prevent effective filtering. Clean tables are smaller and faster to query. |
| **Reproducibility** | Codified preparation steps can be re-run on new data with consistent results — essential for periodic reporting. |
| **Collaboration** | A well-prepared dataset with standardised names and documented grain is immediately usable by anyone on the team. |

## Summary

Before writing a single analytical query, work through these steps:

1. **Combine your sources** — `UNION ALL` with schema alignment, tag each source, validate row counts
2. **Reconcile schema drift** — unify renamed columns, extract temporal dimensions
3. **Remove exact duplicates** — a mechanical first step requiring no domain knowledge
4. **Understand the grain** — check cardinality, identify keys, validate foreign key relationships
5. **Resolve natural key conflicts** — apply a clear business rule to choose which row survives
6. **Standardise relentlessly** — normalise case, trim whitespace, map variant labels, unify nulls

### Notebook roadmap

| Notebook | What you'll learn |
| --- | --- |
| **02 — Medallion Architecture** | The bronze → silver → gold pattern and how the DfE implements it |
| **03 — Combining Snapshots** | Combine termly bronze extracts into `silver.schools_combined` and `silver.pupils_combined` |
| **04 — Reconciling Schema Drift** | Unify renamed columns and extract `term` and `year` |
| **05 — Understanding Your Data** | Remove exact duplicates, check cardinality, keys, and referential integrity |
| **06 — Removing Duplicates** | Resolve natural key conflicts using business rules and `ROW_NUMBER` |
| **07 — Cleaning and Standardising** | Standardise labels, parse JSON, cast types, handle missing data |
| **08 — Building the Gold Layer** | Promote clean tables to gold with documentation |